In [ ]:
####################### N O  T O C A R ############################################
%load_ext autoreload
%autoreload 2
import sys
import os

# Agrega la ruta raíz del proyecto si no está
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# import Configs.configuracion_general as config
import services.Carga.cargar_datos_csv as carga 
from services.Utils.utilidades import *
from services.Correctores.corrector_utils import *
from configs.manager_diccionario_variables import * 
from services.Graficado.graficar_series_y_guardar import graficar_series_y_guardar
from services.Graficado.graficar_mapa_de_posiciones import graficar_mapa_de_posiciones


##################################################################################

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


ModuleNotFoundError: No module named 'services.Graficado.graficar_mapa_con_posiciones'

In [2]:
rutas_de_sondas, seriales_encontrados = carga.buscar_nombre_de_archivo_de_sonda()
rutas_de_sondas

La sonda 4909282 no tiene un archivo CSV en la carpeta de datos crudos.


['C:\\Users\\Atmosfera\\Desktop\\datos_crudos\\doris\\todos_los_datos\\datos_Localizacion_4857577_TOTAL.csv',
 'C:\\Users\\Atmosfera\\Desktop\\datos_crudos\\doris\\todos_los_datos\\datos_Localizacion_4912199_TOTAL.csv',
 'C:\\Users\\Atmosfera\\Desktop\\datos_crudos\\doris\\todos_los_datos\\datos_Localizacion_4912171_TOTAL.csv',
 'C:\\Users\\Atmosfera\\Desktop\\datos_crudos\\doris\\todos_los_datos\\datos_Localizacion_4907604_TOTAL.csv']

In [3]:
diccionario_de_datos_de_sondas = carga.cargar_datos_de_sonda(rutas_de_sondas, seriales_encontrados)

Datos cargados correctamente para la sonda: 4857577
Datos cargados correctamente para la sonda: 4912199
Datos cargados correctamente para la sonda: 4912171
El archivo CSV de la sonda 4907604 está vacío. Se omite esta sonda.


In [4]:
# diccionario_de_datos_de_sondas.keys()

In [5]:
# para 10.3
diccionario_de_sondas_en_fechas = carga.seleccionar_rango_de_fechas(diccionario = diccionario_de_datos_de_sondas, buscar_fechas_anteriores_al_estudio = False)
datos_de_sondas_sin_duplicados = carga.buscar_y_eliminar_duplicados(diccionario_de_sondas_en_fechas)
datos_ordenados = carga.ordernar_datos_por_fecha(datos_de_sondas_sin_duplicados)

Datos ordenados por fecha para la sonda 4857577.
Datos ordenados por fecha para la sonda 4912199.
Datos ordenados por fecha para la sonda 4912171.


c:\programacion\codigos_python\drift_buoys\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [6]:
datos_ordenados.keys()

dict_keys(['4857577', '4912199', '4912171'])

In [7]:
for serial in seriales_encontrados:
    if serial in datos_ordenados:
        datos_ordenados[serial]["tspan_rounded"] = datos_ordenados[serial]["tspan_de_envio"]


In [8]:
# datos_ordenados["4857577"]

In [9]:
# Eliminar datos espurios (solo se revisa si hay valores de rapidez superiores a 2 m/s y se elimina toda la fila)
datos_finales = eliminar_datos_espurios(datos_ordenados)

In [10]:
# Agregar componentes de la velocidad al diccionario con los dataframe de cada sonda
datos_finales = carga.agregar_componentes_de_la_velocidad(datos_finales)

In [11]:
tabla_de_porcentajes = calcular_porcentaje_de_datos_recibidos(datos_finales)

No se encontraron datos para la sonda 4909282.
No se encontraron datos para la sonda 4907604.


c:\programacion\codigos_python\drift_buoys\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [12]:
# datos_finales

In [13]:

tabla_de_porcentajes

,serial_de_sonda,fecha_de_inicio,fecha_final,cantidad_de_datos_esperados,cantidad_de_datos_recibidos,porcentaje_de_datos_recibidos
0,4857577,2026-06-27 11:38:00,2026-06-30 23:44:00,169,135,79.88
1,4909282,2026-06-27 11:38:00,2026-06-30 23:30:00,169,0,0.00
2,4912199,2026-06-27 11:38:00,2026-06-30 23:38:00,169,122,72.19
3,4912171,2026-06-27 11:38:00,2026-06-30 23:54:00,169,119,70.41
4,4907604,2026-06-27 11:38:00,2026-06-30 23:30:00,169,0,0.00


In [ ]:
graficar_series_y_guardar(datos= datos_finales, mostrar_figura=False)



Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202606\SONDA_OCEANOGRAFICA_NS_4857577_REALT_20260628_20260630.png
Advertencia: Serial 4909282 no encontrado en los datos
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202606\SONDA_OCEANOGRAFICA_NS_4912199_REALT_20260628_20260630.png
Figura guardada correctamente en C:\Users\Atmosfera\Desktop\datos_procesados\doris\10.3\202606\SONDA_OCEANOGRAFICA_NS_4912171_REALT_20260628_20260630.png
Advertencia: Serial 4907604 no encontrado en los datos


In [ ]:
graficar_mapa_con_posiciones(datos=datos_finales, mostrar_figura=False)

In [ ]:
# ruta_a_carpeta = crear_ruta_a_carpeta(get_carpeta_guardado_datos_procesados())
# ruta_pickle = os.path.join(ruta_a_carpeta, get_nombre_archivo_datos_procesados())
# diccionario = cargar_diccionario_pickle(ruta_pickle)

# Guardar datos del estudio en archivo .pkl
carpeta_de_destino = crear_ruta_a_carpeta(get_carpeta_guardado_datos_procesados())
nombre_de_archivo = get_nombre_archivo_datos_procesados()
guardar_diccionario_como_pickle(diccionario = datos_finales, 
                                ruta = carpeta_de_destino, 
                                nombre_archivo=nombre_de_archivo)

# Guardar Tabla de porcentajes
ruta_a_carpeta_de_datos_procesados = crear_ruta_a_carpeta(get_carpeta_guardado_datos_procesados())
guardar_porcentajes_en_excel(data= tabla_de_porcentajes, ruta= ruta_a_carpeta_de_datos_procesados, nombre_de_archivo=get_nombre_del_excel_de_porcentajes())